# Figure 8 — p_L Sensitivity & Depth Heatmap
Reproduces Figure 8 of Huang et al. 2016.

1. Run **Mount & Copy** cell first.
2. Edit the config cell for your tab.
3. Run the training loop cell.

> Runtime → Change runtime type → **A100 GPU**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
ROOT = '/content/drive/MyDrive/stoch_depth'

for fname in ['model.py', 'data.py', 'train.py', 'evaluate.py']:
    shutil.copy(f'{ROOT}/code/{fname}', f'/content/{fname}')
    print(f'Copied {fname}')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Figure 8 Left — p_L Sweep
Edit config then run the loop.

In [ ]:
# ── EDIT FOR YOUR TAB ────────────────────────────────────────────────
# Tab 5: P_L_VALUES=[0.1,0.2,0.3], MODE='linear'
# Tab 6: P_L_VALUES=[0.4,0.5,0.6], MODE='linear'
# Tab 7: P_L_VALUES=[0.7,0.8,0.9], MODE='linear'
# Tab 8: P_L_VALUES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9], MODE='uniform'
P_L_VALUES = [0.1, 0.2, 0.3]
MODE       = 'linear'
# ─────────────────────────────────────────────────────────────────────
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
print(f'Config: p_L={P_L_VALUES}  mode={MODE}')

In [ ]:
import os
for p_L in P_L_VALUES:
    tag  = f'cifar10_pL{p_L}_mode{MODE}_n18'
    ckpt = f'{ROOT}/fig8_left/{MODE}/checkpoints/{tag}_last.pt'
    resume = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
    out = f'{ROOT}/fig8_left/{MODE}'
    print(f'\n=== p_L={p_L} mode={MODE} ===')
    os.environ['RESUME'] = resume
    os.environ['OUT']    = out
    os.environ['PL']     = str(p_L)
    os.environ['MODE']   = MODE
    !python /content/train.py \
        --dataset cifar10 --p_L $PL --survival_mode $MODE \
        --epochs 500 \
        --data_root /content/data \
    --num_workers 2 \
        --out_dir $OUT \
        $RESUME
print('\nAll done.')

## Figure 8 Right — Depth Heatmap
Edit config then run the loop.

In [ ]:
# ── EDIT FOR YOUR TAB ────────────────────────────────────────────────
# Tab 9:  BLOCKS=[3,6]   → depths 20L, 38L
# Tab 10: BLOCKS=[9,12]  → depths 56L, 74L
# Tab 11: BLOCKS=[15,18] → depths 92L, 110L
BLOCKS = [3, 6]
# ─────────────────────────────────────────────────────────────────────
import os
ROOT = '/content/drive/MyDrive/stoch_depth'
P_L_VALUES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print(f'Config: blocks={BLOCKS}, depths={[6*n+2 for n in BLOCKS]}')

In [ ]:
import os
for n in BLOCKS:
    depth = 6 * n + 2
    for p_L in P_L_VALUES:
        mode = 'constant' if p_L == 1.0 else 'linear'
        tag  = f'cifar10_pL{p_L}_mode{mode}_n{n}'
        ckpt = f'{ROOT}/fig8_right/checkpoints/{tag}_last.pt'
        resume = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
        out = f'{ROOT}/fig8_right'
        print(f'\n=== depth={depth}L  p_L={p_L} ===')
        os.environ['RESUME'] = resume
        os.environ['OUT']    = out
        os.environ['PL']     = str(p_L)
        os.environ['MODE']   = mode
        os.environ['N']      = str(n)
        !python /content/train.py \
            --dataset cifar10 --p_L $PL --survival_mode $MODE \
            --blocks_per_group $N \
            --epochs 500 \
            --data_root /content/data \
    --num_workers 2 \
            --out_dir $OUT \
            $RESUME
print('\nAll done.')